# Import libraries

In [9]:
import pandas as pd
import numpy as np

# ML Models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,classification_report

# Pre-Processing
import re

# NLTK related libraries
import nltk
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Count vectorization
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder

from sklearn.feature_extraction.text import TfidfVectorizer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Data

In [38]:
data = pd.read_csv("imdb_reviews.csv").sample(2000) # sample taken to reduce processing time (but ideally need to take all data)
data.head()

,review,sentiment
41249,I'm afraid that I didn't like this movie very ...,negative
9191,"As a movie, THE ITALIAN JOB is ok at best; goo...",positive
40755,I've already commented on this film (under the...,negative
7155,"Well, my goodness, am I disappointed. When I f...",negative
23500,Tom and Jerry are transporting goods via airpl...,negative


# Pre-Processing

In [39]:
sw = stopwords.words("english")
ps = PorterStemmer()

In [40]:
sw

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [41]:
def preprocessing(text):
    text = re.sub("[^a-zA-Z0-9 ]","",text) # except english alphabets, numbers and space remove everything
    text = text.lower() # update all letters to lowercase
    text = nltk.word_tokenize(text) # separate each word as a single token
    text = [ word for word in text if word not in sw ] # remove stopwords
    text = [ ps.stem(word) for word in text] # stemming (based on 13-14 rules)
    return " ".join(text) # join the tokenized words again

In [42]:
preprocessing("I love food so much")

'love food much'

In [43]:
data["updated_review"] = data["review"].apply(preprocessing)
encoder = LabelEncoder()
data["sentiment"] = encoder.fit_transform(data["sentiment"])

In [44]:
data.head()

,review,sentiment,updated_review
41249,I'm afraid that I didn't like this movie very ...,0,im afraid didnt like movi much apart save grac...
9191,"As a movie, THE ITALIAN JOB is ok at best; goo...",1,movi italian job ok best good great actingbr b...
40755,I've already commented on this film (under the...,0,ive alreadi comment film name thelegendarywd s...
7155,"Well, my goodness, am I disappointed. When I f...",0,well good disappoint first heard news remak ro...
23500,Tom and Jerry are transporting goods via airpl...,0,tom jerri transport good via airplan africa wh...


# Dividing data into train and test

In [45]:
x_train,x_test,y_train,y_test = train_test_split(data["updated_review"],data["sentiment"],test_size=0.2)

# Vectorization

In [46]:
cv = CountVectorizer(max_features=20000) # it will take max 20000 unique words
x_train_vectors = cv.fit_transform(x_train).toarray()
x_test_vectors = cv.transform(x_test).toarray()

In [47]:
x_train_vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0]])

In [48]:
x_test_vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [93]:
cv.get_feature_names_out()

array(['007', '010', '02', ..., 'zulu', 'zwart', 'zwick'], dtype=object)

# Model Selection

In [49]:
models=[LogisticRegression(),RandomForestClassifier(n_estimators=200),DecisionTreeClassifier(),KNeighborsClassifier(n_neighbors=5)]

for model in models:
    model.fit(x_train_vectors,y_train)
    y_pred = model.predict(x_test_vectors)
    print("==="*20)
    print(model)
    print(classification_report(y_test,y_pred))

LogisticRegression()
              precision    recall  f1-score   support

           0       0.85      0.84      0.85       187
           1       0.86      0.87      0.87       213

    accuracy                           0.86       400
   macro avg       0.86      0.86      0.86       400
weighted avg       0.86      0.86      0.86       400

RandomForestClassifier(n_estimators=200)
              precision    recall  f1-score   support

           0       0.84      0.79      0.81       187
           1       0.83      0.86      0.84       213

    accuracy                           0.83       400
   macro avg       0.83      0.83      0.83       400
weighted avg       0.83      0.83      0.83       400

DecisionTreeClassifier()
              precision    recall  f1-score   support

           0       0.66      0.65      0.65       187
           1       0.70      0.70      0.70       213

    accuracy                           0.68       400
   macro avg       0.68      0.68      0.

# Final Model

In [50]:
final_model = LogisticRegression()
final_model.fit(x_train_vectors,y_train)

LogisticRegression()

# Decide New Review Sentiment

In [90]:
text = "very bad movie it is."
text = preprocessing(text)
vector = cv.transform([text]).toarray()
final_model.predict(vector)

array([0])

In [91]:
encoder.inverse_transform([0,1])

array(['negative', 'positive'], dtype=object)

In [92]:
encoder.inverse_transform(final_model.predict(vector))

array(['negative'], dtype=object)